# 🗑️ Marine Pollution Object Detection 🤖
Please check `READ.ME` file for prerequisite installations before continuing

## 🌱 Preparing the Environment

In [1]:
# Run GPU acceleration if available
import torch

print("ROCm / CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))


ROCm / CUDA Available: True
GPU Device Name: AMD Radeon RX 7700 XT


## 🗂️ Loading the Dataset
The dataset includes `189` images, annotated in `YOLOv11` format.
  
The `7` types of classification include:
1. Bottle Cap
2. Cone Cap
3. Leaf
4. Netting
5. Plastic Bag
6. Plastic Strand
7. Sponge

Dataset was <font color='#FFC067'>split</font> and <font color='#FFC067'>pre-processed</font> for consistency using `Roboflow`:
1. Auto-orientation of pixel data (with EXIF-orientation stripping)  
2. Resized to `512x512` (Stretch)
  


### 📢Note‼️
* Running the subparts below are not required if you directly downloaded `Trash-Tank-YOLOv11.zip` and do not want to use a database. 

### Part 1: 💾 Importing from Roboflow
* Replace `ROBOFLOW_API_KEY` with your own in `.env`
* Data is already divided into training, validation, and test sets
* Check for correct dataset path

In [2]:
import os
from dotenv import load_dotenv
from roboflow import Roboflow

load_dotenv()

rf_key = os.getenv('ROBOFLOW_API_KEY')
rf = Roboflow(api_key=rf_key)

project = rf.workspace('brittany-nguyen').project('trash-tank-yolov11')
version = project.version(4)
dataset = version.download('yolov11')

loading Roboflow workspace...
loading Roboflow project...


In [3]:
# Verify dataset path
try:
    dataset_path = dataset.location
    print(f"Files in {os.path.basename(dataset_path)}: {os.listdir(dataset_path)}")

    # Set file necessary file paths
    train_path = os.path.join(dataset_path, 'train/')
    val_path = os.path.join(dataset_path, 'val/')
    yaml_path = os.path.join(dataset_path, 'data.yaml')

    print("All paths are set successfully!")

except Exception as e:
    print(f"\nUnable to locate file path. Error details:\n{e}")

Files in Trash-Tank-YOLOv11-4: ['data.yaml', 'README.roboflow.txt', 'valid', 'README.dataset.txt', 'train', 'test']
All paths are set successfully!


### Part 2: 🔌 Connecting to the DATABASE
Configure the following parameters in `env`:
* `DRIVER`: Specify the exact driver software name 
* `SERVER`: Use network name or IP address of the machine
* `DATABASE`: Choose the specific database for the project
    * e.g. Create a database inside `master` 
* Choose your usual authentication used to access the database



In [6]:
# Connect to SQL Server database
import pyodbc
import os 
from dotenv import load_dotenv

load_dotenv()

db_server = os.getenv('DB_SERVER')
db_username = os.getenv('DB_USERNAME')
db_password = os.getenv('DB_PASSWORD')

try:
    conn_str = (
        "DRIVER={ODBC Driver 18 for SQL Server};"
        f"SERVER={db_server};"
        "DATABASE=MarinePollutionDB;"
        f"UID={db_username};" 
        f"PWD={db_password};" 
        "TrustServerCertificate=yes;"
    )

    conn = pyodbc.connect(conn_str)
    cursor = conn.cursor()
    print("Successfully connected to the database!\n")

    
except Exception as e:
    print(f"\nConnection failed. Error details:\n{e}")


Successfully connected to the database!



### Part 3: 🛢️ Initializing Schema Infrastructure
* Uses the script `db.py` and `schema.sql` to create tables and store procedures
* Please ensureall metadata are loaded corectly


In [7]:
from db import initialize_schema

initialize_schema(cursor)
conn.commit()

print("Schema infrastracture created.")

Schema infrastracture created.


### Part 4: 📦 Storing Image and Label Annotations
* Refer to `README.md` or `ERD.png` for database architecture
* Convert `dataset.yaml` for `classId` and `className`
* Use stored procedures to load all image data

In [10]:
from db import load_class

try:
    load_class(cursor, yaml_path)
    conn.commit()
    print("Classes loaded successfully.")

except Exception as error:
    conn.rollback()
    print(f"Class loading failed; changes rolled back: {error}\n")
    print("Check if classes were loaded correctly before deleting and restarting.")

Class loading failed; changes rolled back: ('23000', "[23000] [Microsoft][ODBC Driver 18 for SQL Server][SQL Server]Violation of PRIMARY KEY constraint 'PK__Class__7577345EF5E685B8'. Cannot insert duplicate key in object 'dbo.Class'. The duplicate key value is (0). (2627) (SQLExecDirectW)")

Check if classes were loaded correctly before deleting and restarting.


In [ ]:
IMAGE_EXTENSION = ('.jpg', '.jpeg', '.png', '.bmp', 'tif', 'tiff')

# Correctly maps folder names
SPLIT_ID = {
    'train': 0,
    'val': 1,
    'valid': 1,
    'validation': 1,
    'test': 2,
}

In [ ]:
cursor.close()
conn.close()

## 🧩 Augmenting Image Data
Apply transformations <font color='#FFC067'>only to the training set</font>:
* Adjust `p` to reduce overfitting 
  
Keep validation and test sets unaugmented:
* Properly measures model performance on real-world data

Augmentation used:
1. Random Horizontal Flip 
2. Random Rotation `45` degrees
3. Random Crop into `128x128px` square
4. Standardized Color Values 

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
import os

# Define augmentation pipeline
transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=45, p=0.5),
    A.RandomCrop(width=128, height=128),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(), # Convert final image into PyTorch tensor
])


# Process valid file paths
input_dir = os.path.join(dataset_path, 'train/images')
output_dir = os.path.join(dataset_path, 'augmented/images')


def augment_images(input_dir, output_dir, transform):

    # Create subfolder for augmented images
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    
    for image_file in os.listdir(input_dir):
        image_path = os.path.join(input_dir, image_file)
        image = cv2.imread(image_path)
        
        # Apply augmentation
        augmented = transform(image=image)['image']
        
        # Save augmented image
        output_path = os.path.join(output_dir, image_file)
        cv2.imwrite(output_path, augmented.numpy().transpose(1, 2, 0) * 255)  # Convert back to original range

augment_images(input_dir, output_dir, transform)

## 🚀 Initializing TensorBoard
TensorBoard is a <font color='#FFC067'>visualization tool</font> for real-time insights of the training model:
* Monitors key metrics (e.g. loss, accuracy, and learning rates)
* Adjusts in real-time and improves overall model performance
* Showcases possible overfitting or underfitting cases

Note: <font color='#FFC067'>Frequently refresh</font> the button in the top right of the TensorBoard.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/runs/detect/train

## 🎯 Training the YOLOv11 Model
Key Training Parameters:
* `imgsz` defines the <font color='#FFC067'>target image size</font> of the inputs for consistency
* `batch` processes the <font color='#FFC067'>number of images</font> concurrently  
* `epochs` establish the <font color='#FFC067'>number of complete cycles</font> for training

Note: It is important to check the accuracy, find class weaknesses, and track improvements within the model.
  

In [ ]:
# Load YOLOv11 model
model = YOLO('yolo11n.pt')

# Set dataset configuration to YAML file
dataset_config = os.path.join(dataset_path, 'data.yaml') # Load all class labels

# Train the model
results = model.train(
    data=dataset_config,
    epochs=100,
    batch=64,  # Set appropriate batch size
    imgsz=640,  # Standardize image size for training
    plots=True,
    patience=50
)

# Inspect training results
print(results)

## 📊 Evaluating the Model
* Use validation data and metrics to evaluate the model performance

In [ ]:
from IPython.display import Image, display
import os

# Set the base directory
base_dir = os.path.join(dataset_path, 'runs/detect/train')

# List of filenames to display
filenames = [
    "labels.jpg",
    "F1_curve.png",
    "PR_curve.png",
    "P_curve.png",
    "R_curve.png",
    "confusion_matrix.png",
    "confusion_matrix_normalized.png"
]

# Display each image
for filename in filenames:
    image_path = os.path.join(base_dir, filename)
    display(Image(image_path))

## 🔎 Running an Inference
* <font color = "FFC067">Test real-world application</font> for the model
  
Configure the two inference parameters as needed: `conf` and `iou`
  * <font color = "FFC067">Confidence Threshold</font> (conf):
    * Sets minimum probability score required for <font color = "FFC067">keeping detections</font>  and predicted bounding box
    * Adjust appropriately to reduce either low-confidence guesses or false positives
  * <font color = "FFC067">Intersection over Union Threshold</font> (iou):
    * Sets Non-Maximum Suppression (NMS) value required for <font color = "FFC067">removing duplicates</font> and overlapping bounding boxes
    * Adjust appropriately to control the strength of merging and deleting neighboring duplicate detections

In [ ]:
# Download the video to test the model

!wget https://huggingface.co/datasets/OceanCV/PlasticTank_Video/resolve/main/tankvid.mp4?download=true -O tankvid.mp4

In [ ]:
import cv2
from ultralytics import YOLO

model_path = os.path.join(base_dir, 'weights/best.pt')
model = YOLO(model_path)

video_path = 'tankvid.mp4'

results = model.track(source=video_path, save=True, show=True, conf=0.25, iou=0.7)